# AI-RAG · Week 12 — Notebook 2 (Day 2)
## The M2 Milestone & Design Review #2

Package six weeks of Phase 2 into a **production-ready, defensible** RAG system, then prepare to defend it at **DR #2**. This half is more packaging and process than code: the runnable piece is the M2 **readiness checker**; the rest is the deliverable structure — the 8-KPI dashboard, the ADR consolidation, and the DR #2 one-pager. Runs **offline**.

> The checker code is verbatim from `scripts/m2_check.py`; setup writes example M2 documents so it runs green (replace them with your own).

### Setup

In [1]:
# --- notebook setup (delivery glue only) -------------------------------------
# m2_check reads the M2 documents from ./docs. We write example versions here so the
# checker runs green and you can see what a complete M2 looks like; replace them with
# your own. Everything is offline (stdlib only).
from pathlib import Path
docs = {
    "docs/kpi/wk12-snapshot.md":            r"""# wk12-snapshot.md — M2 · Production RAG (Phase 2 close)

> *The M2 milestone KPI dashboard. Phase 2 is done: the assistant is measured,
> debugged, and defensible. Numbers below are a filled example — replace with
> your own live run before DR #2. This is portfolio material: six snapshots from
> wk6 to wk12 showing real engineering progress.*

| # | KPI | wk12 | Source | vs wk11 |
|---|---|---|---|---|
| 1 | Task success rate | 0.92 | W5 judge (live) | ▲ 0.90 → 0.92 |
| 2 | Grounded response rate | 0.92 | RAGAS faithfulness | ▲ 0.91 → 0.92 |
| 3 | Hallucination rate | 0.08 | 1 − faithfulness | ▼ 0.09 → 0.08 (good) |
| 4 | Retrieval hit rate | 0.90 | correct chunk in top-K | reconciled — see note |
| 5 | Tool success rate | N/A (W13 — no tools yet) | — | unchanged (N/A) |
| 6 | Cost / query (warm) | $0.000131 | results.db | unchanged |
| 7 | Latency p50 / p95 | 184 / 312 ms | results.db | unchanged |
| 8 | Escalation rate | N/A (W17 — no HITL yet) | — | unchanged (N/A) |

**Supporting RAGAS metrics:** faithfulness 0.92 · answer relevance 0.93 · context
precision 0.85 · context recall 0.90. **KB version:** `v2.w10` · **golden set:** 20
questions · **eval mode:** live RAGAS.

---

## ⚠️ KPI #4 reconciliation (W11 → W12)

The programme defines KPI #4 (retrieval hit rate) as *"% of queries where the
correct chunk appears in top-K."* W11 reported it via RAGAS **context precision**
(0.82) as a stand-in. From M2 onward, KPI #4 is measured to its **canonical
definition** — the fraction of golden questions with the correct chunk in the
top-K — which reads **0.90**. Context **precision** (0.85) is retained as a
*supporting* retrieval-quality metric, not KPI #4 itself. This is a change of
definition, not a regression; compare KPI #4 to the canonical measure from here.

---

## W6 → W12 trajectory

| KPI | wk6 | wk7 | wk8 | wk9 | wk10 | wk11 | wk12 |
|---|---|---|---|---|---|---|---|
| Task success | 0.72 | 0.78 | 0.83 | 0.86 | 0.88 | 0.90 | **0.92** |
| Grounded (judge→RAGAS) | 0.80* | 0.85* | 0.90* | 0.93* | 0.94* | 0.91‡ | **0.92** |
| Retrieval hit rate | 0.70 | 0.78 | 0.85 | 0.91 | 0.91 | 0.82§ | **0.90** |
| Cost / query (warm) | $.0009 | $.0007 | $.0006 | $.0004 | $.00013 | $.00013 | **$.00013** |
| Latency p50 (ms) | 1100 | 950 | 870 | 785 | 184 | 184 | **184** |

`*` groundedness by the W5 judge. `‡` first RAGAS faithfulness measurement. `§`
W11 reported KPI #4 as context precision; wk12 reconciles to the canonical hit
rate. Do not read the wk10→wk11 steps on the starred/marked rows as regressions —
they are instrument/definition changes (documented in the ADR).

## KPI trajectory commentary

- **What improved most:** cost and latency, by an order of magnitude at wk10
  (semantic cache + tombstones). On quality, task success climbed steadily across
  the build weeks (wk6→wk9) and ticked up again at wk12 after a debugging fix.
- **What drove the wk12 gains:** the RAG debugging work. Isolating a multi-hop
  **recall miss** (drill bucket: retrieval) and raising top-K for those questions
  lifted retrieval hit rate and, with it, task success and groundedness. One
  well-diagnosed retrieval bug moved three KPIs.
- **What's still N/A:** tool success (#5) and escalation (#8) — the assistant has
  no tools (arrive W13) and no human-in-the-loop (arrives W17). Reporting a number
  there would be fabrication; they become real in Phase 3+.
- **Cache health:** cache-hit accuracy (0.92) remains at or above cache-miss
  accuracy (0.88) — the W10 cache is a cost win, not a correctness risk.

---

## M2 status

Production-ready RAG: FastAPI service (W5) wrapping the full W6–W11 pipeline,
`pytest eval/` green against baseline thresholds, all 8 KPIs reported (6 numeric,
2 honest N/A), architecture consolidated in ADR v2 (M2). Ready to defend retrieval
+ KB-lifecycle decisions at DR #2.

**Next snapshot:** M3 (W18) — after Phase 3 adds tools/agents (KPI #5 becomes
real) and injection defenses.""",
    "docs/adr/0001-capstone-framing.md":    r"""# ADR 0001 — Capstone Framing — v2 (M2)

*(Prior W1–W11 entries elided — keep your repo's full text. This is the Phase-2
consolidation the milestone asks for: every decision in one table.)*

## Phase 2 consolidated decisions (v2 · M2)

| Area | Decision | Why |
|---|---|---|
| Chunking | 512-token, 64 overlap | balances recall vs precision |
| Hybrid weight | BM25 0.4 + dense | catches exact-term queries |
| Reranker + top-K | cross-encoder, top-K 3 | lifts context precision; small prompt |
| Cache | semantic (0.95) + tombstones | cost win, safe deletes |
| Eval | RAGAS thresholds (faithfulness 0.85, recall 0.80, ...) | defensible bar |
| KPIs | 8-KPI dashboard, kb_version-tagged | portfolio evidence |

Status: **v2 (M2)** — Phase 2 closed; retrieval + KB-lifecycle decisions consolidated.
""",
    "docs/dr/dr2-summary-EXAMPLE.md":        r"""# DR #2 — EXAMPLE — Enterprise Knowledge Assistant — Retrieval + KB Lifecycle Defense

> Filled example for the capstone use case. Yours will differ; match the depth.

## 1. Problem framing
The assistant answers employees' questions about HR/IT/Finance policies from an
internal document corpus. Constraints: answers must be **grounded** (no invented
policy), cheap at scale, and safe when a topic isn't covered. RAG is the right
approach: the policy corpus changes often and must be cited — fine-tuning would
bake in stale facts and can't cite sources.

## 2. Retrieval choices  (trade-offs with KPI evidence)
- **Hybrid, BM25 weight 0.4.** Dense retrieval alone missed exact-term queries
  (policy codes, section numbers); adding BM25 raised retrieval hit rate. 0.4 was
  the point where hit rate peaked without precision dropping.
- **Cross-encoder rerank, top-K = 3.** Reranking the top candidates lifted context
  precision from ~0.7 to 0.85; top-K = 3 keeps the prompt small (cost) while
  covering multi-hop answers after the S1 recall fix.
- **Rejected:** dense-only (missed exact terms); top-K = 5 (added ~600 prompt
  tokens for no measurable quality gain).

## 3. KB lifecycle  (failure handling)
- **Tombstone policy:** deletes set `deleted_at` and are filtered from retrieval;
  we never hard-delete, so eval runs stay reproducible.
- **Update workflow:** full re-ingest, tombstone the old chunks, bump
  `kb_version` — see `docs/runbooks/rag_debugging.md` and the W10 update runbook.
- **Cache invalidation:** stale-cache is a real failure mode (drill S5); cache
  entries are invalidated on `kb_version` change. Cache-hit accuracy (0.92) stays
  above cache-miss accuracy (0.88), so the cache isn't serving stale answers.

## 4. The 8-KPI dashboard  (KPI reporting, with trend)
Task success **0.92**, grounded **0.92**, hallucination **0.08**, retrieval hit
**0.90**, cost **$0.000131**, latency **184/312 ms**; tool success and escalation
**N/A** (arrive W13 / W17). Full wk6→wk12 trajectory in `docs/kpi/wk12-snapshot.md`
— quality climbed across the build weeks and again at wk12 after a retrieval fix;
cost/latency dropped an order of magnitude at wk10.

## 5. Open trade-offs & questions
- **Open trade-off:** the semantic-cache threshold (0.95) was tuned on only 20
  golden questions; at real traffic I'd tune it against a measured false-hit rate.
- **Open question for reviewers:** is 20 golden questions enough to trust the KPI
  trend, or should the golden set grow before Phase 3 adds tool-use complexity?""",
    "docs/runbooks/rag_debugging.md":        r"""# RAG Debugging Runbook

Reproduce -> Narrow (get the trace) -> Isolate (first stage whose output is wrong)
-> Fix (smallest change for that bucket; treat retrieval changes as candidates you
test against the golden set) -> Verify (re-run the whole eval suite).
""",
    "docs/debugging/rag_failure_taxonomy.md": r"""# RAG Failure Taxonomy — four primary runtime buckets

retrieval · generation · prompt · infra — these localise a *runtime* failure from a
trace. Beyond the runtime pipeline, cross-cutting failures — ingestion/data-quality,
access & security, evaluation/reference defects, and user-input/policy — live upstream
or across stages and are diagnosed elsewhere (W8, W11).
""",
}
for rel, text in docs.items():
    p = Path(rel); p.parent.mkdir(parents=True, exist_ok=True); p.write_text(text)
print("wrote", len(docs), "example M2 documents under ./docs/")


wrote 5 example M2 documents under ./docs/


## 1 · The five M2 items (non-negotiable)

| # | Item | What it is |
|---|---|---|
| 1 | **Production-ready RAG API** | FastAPI (W5) wrapping the W6–W11 pipeline, running end-to-end |
| 2 | **Full eval suite** | your `eval/` folder, pytest-runnable, all thresholds pass |
| 3 | **8-KPI dashboard** | `wk12-snapshot.md` — all 8 KPIs + the wk6→wk12 trajectory |
| 4 | **ADR v2 (M2)** | every Phase-2 decision consolidated in one place |
| 5 | **DR #2 one-pager** | the retrieval + KB-lifecycle defense |

Items 1–2 are your own capstone code (verified with the commands the checker prints); items 3–5 are documents the checker verifies directly.

## 2 · The readiness checker — `scripts/m2_check.py`

Verifies the three M2 documents and prints the commands for the two code items. Offline.

In [2]:
#!/usr/bin/env python3
"""scripts/m2_check.py — is your M2 deliverable in shape?

M2 has five items. Two of them (the running API, the passing eval suite) are your
own capstone code and are checked with the commands this script prints. The other
three are documents this script verifies directly: the KPI snapshot, the ADR
consolidation, and the DR #2 one-pager. Runs offline.

    python3 scripts/m2_check.py
"""
from __future__ import annotations

import re
import sys
from pathlib import Path

ROOT = Path.cwd()   # notebook: resolve doc paths from the working directory
GREEN, RED, YEL, NC = "\033[32m", "\033[31m", "\033[33m", "\033[0m"


def _read(rel: str) -> str:
    p = ROOT / rel
    return p.read_text() if p.exists() else ""


def check(label: str, ok: bool, hint: str = "") -> bool:
    mark = f"{GREEN}✓{NC}" if ok else f"{RED}✗{NC}"
    print(f"  {mark} {label}" + (f"  — {hint}" if not ok and hint else ""))
    return ok


def main() -> int:
    print("M2 deliverable — document checks (items 3–5):\n")
    ok = True

    snap = _read("docs/kpi/wk12-snapshot.md")
    kpi_rows = all(re.search(rf"^\|\s*{n}\s*\|", snap, re.M) for n in range(1, 9))
    ok &= check("KPI snapshot: docs/kpi/wk12-snapshot.md exists", bool(snap),
                "write the 8-KPI dashboard")
    ok &= check("KPI snapshot: all 8 KPI rows present", kpi_rows)
    ok &= check("KPI snapshot: wk6→wk12 trajectory + commentary",
                "trajectory" in snap.lower() and "commentary" in snap.lower())

    adr = _read("docs/adr/0001-capstone-framing.md")
    ok &= check("ADR: docs/adr/0001-capstone-framing.md exists", bool(adr))
    ok &= check("ADR: marked v2 (M2)", "v2 (m2)" in adr.lower() or "v2 (M2)" in adr)
    ok &= check("ADR: consolidates Phase 2 (retrieval + lifecycle + eval)",
                all(k in adr.lower() for k in ("rerank", "tombstone", "ragas")))

    dr = _read("docs/dr/dr2-summary-EXAMPLE.md") or _read("docs/dr/dr2-summary-template.md")
    ok &= check("DR #2 one-pager: docs/dr/dr2-summary-*.md exists", bool(dr))
    ok &= check("DR #2 one-pager: covers retrieval + KB lifecycle + KPIs",
                all(k in dr.lower() for k in ("retriev", "lifecycle", "kpi")))

    rb = _read("docs/runbooks/rag_debugging.md")
    tax = _read("docs/debugging/rag_failure_taxonomy.md")
    ok &= check("Debugging runbook present", bool(rb))
    ok &= check("Failure taxonomy present", bool(tax))

    print(f"\nItems 1–2 are your capstone code — verify them with:\n"
          f"  {YEL}# 1. Production API (run your service, then):{NC}\n"
          f"  curl -s localhost:8000/health | python3 -m json.tool\n"
          f"  curl -s -XPOST localhost:8000/ask -d '{{\"question\":\"What is the leave policy?\"}}'\n"
          f"  {YEL}# 2. Eval suite (your W11 eval/ folder):{NC}\n"
          f"  python3 -m pytest eval/ -q\n")

    if ok:
        print(f"{GREEN}M2 documents complete.{NC} Finish items 1–2 above, then you're ready for DR #2.")
        return 0
    print(f"{RED}Some M2 documents are missing or incomplete (see ✗ above).{NC}")
    return 1


**Run it** against the example M2 docs we wrote in setup:

In [3]:
rc = main()
print('\nexit code:', rc, '(0 = all document checks passed)')

M2 deliverable — document checks (items 3–5):

  ✓ KPI snapshot: docs/kpi/wk12-snapshot.md exists
  ✓ KPI snapshot: all 8 KPI rows present
  ✓ KPI snapshot: wk6→wk12 trajectory + commentary
  ✓ ADR: docs/adr/0001-capstone-framing.md exists
  ✓ ADR: marked v2 (M2)
  ✓ ADR: consolidates Phase 2 (retrieval + lifecycle + eval)
  ✓ DR #2 one-pager: docs/dr/dr2-summary-*.md exists
  ✓ DR #2 one-pager: covers retrieval + KB lifecycle + KPIs
  ✓ Debugging runbook present
  ✓ Failure taxonomy present

Items 1–2 are your capstone code — verify them with:
  # 1. Production API (run your service, then):
  curl -s localhost:8000/health | python3 -m json.tool
  curl -s -XPOST localhost:8000/ask -d '{"question":"What is the leave policy?"}'
  # 2. Eval suite (your W11 eval/ folder):
  python3 -m pytest eval/ -q

M2 documents complete. Finish items 1–2 above, then you're ready for DR #2.

exit code: 0 (0 = all document checks passed)


## 3 · The 8-KPI dashboard + wk6→wk12 trajectory

The M2 dashboard is portfolio material — six snapshots of real progress. Display the one we wrote:

In [4]:
from IPython.display import Markdown, display
display(Markdown(Path('docs/kpi/wk12-snapshot.md').read_text()))

# wk12-snapshot.md — M2 · Production RAG (Phase 2 close)

> *The M2 milestone KPI dashboard. Phase 2 is done: the assistant is measured,
> debugged, and defensible. Numbers below are a filled example — replace with
> your own live run before DR #2. This is portfolio material: six snapshots from
> wk6 to wk12 showing real engineering progress.*

| # | KPI | wk12 | Source | vs wk11 |
|---|---|---|---|---|
| 1 | Task success rate | 0.92 | W5 judge (live) | ▲ 0.90 → 0.92 |
| 2 | Grounded response rate | 0.92 | RAGAS faithfulness | ▲ 0.91 → 0.92 |
| 3 | Hallucination rate | 0.08 | 1 − faithfulness | ▼ 0.09 → 0.08 (good) |
| 4 | Retrieval hit rate | 0.90 | correct chunk in top-K | reconciled — see note |
| 5 | Tool success rate | N/A (W13 — no tools yet) | — | unchanged (N/A) |
| 6 | Cost / query (warm) | $0.000131 | results.db | unchanged |
| 7 | Latency p50 / p95 | 184 / 312 ms | results.db | unchanged |
| 8 | Escalation rate | N/A (W17 — no HITL yet) | — | unchanged (N/A) |

**Supporting RAGAS metrics:** faithfulness 0.92 · answer relevance 0.93 · context
precision 0.85 · context recall 0.90. **KB version:** `v2.w10` · **golden set:** 20
questions · **eval mode:** live RAGAS.

---

## ⚠️ KPI #4 reconciliation (W11 → W12)

The programme defines KPI #4 (retrieval hit rate) as *"% of queries where the
correct chunk appears in top-K."* W11 reported it via RAGAS **context precision**
(0.82) as a stand-in. From M2 onward, KPI #4 is measured to its **canonical
definition** — the fraction of golden questions with the correct chunk in the
top-K — which reads **0.90**. Context **precision** (0.85) is retained as a
*supporting* retrieval-quality metric, not KPI #4 itself. This is a change of
definition, not a regression; compare KPI #4 to the canonical measure from here.

---

## W6 → W12 trajectory

| KPI | wk6 | wk7 | wk8 | wk9 | wk10 | wk11 | wk12 |
|---|---|---|---|---|---|---|---|
| Task success | 0.72 | 0.78 | 0.83 | 0.86 | 0.88 | 0.90 | **0.92** |
| Grounded (judge→RAGAS) | 0.80* | 0.85* | 0.90* | 0.93* | 0.94* | 0.91‡ | **0.92** |
| Retrieval hit rate | 0.70 | 0.78 | 0.85 | 0.91 | 0.91 | 0.82§ | **0.90** |
| Cost / query (warm) | $.0009 | $.0007 | $.0006 | $.0004 | $.00013 | $.00013 | **$.00013** |
| Latency p50 (ms) | 1100 | 950 | 870 | 785 | 184 | 184 | **184** |

`*` groundedness by the W5 judge. `‡` first RAGAS faithfulness measurement. `§`
W11 reported KPI #4 as context precision; wk12 reconciles to the canonical hit
rate. Do not read the wk10→wk11 steps on the starred/marked rows as regressions —
they are instrument/definition changes (documented in the ADR).

## KPI trajectory commentary

- **What improved most:** cost and latency, by an order of magnitude at wk10
  (semantic cache + tombstones). On quality, task success climbed steadily across
  the build weeks (wk6→wk9) and ticked up again at wk12 after a debugging fix.
- **What drove the wk12 gains:** the RAG debugging work. Isolating a multi-hop
  **recall miss** (drill bucket: retrieval) and raising top-K for those questions
  lifted retrieval hit rate and, with it, task success and groundedness. One
  well-diagnosed retrieval bug moved three KPIs.
- **What's still N/A:** tool success (#5) and escalation (#8) — the assistant has
  no tools (arrive W13) and no human-in-the-loop (arrives W17). Reporting a number
  there would be fabrication; they become real in Phase 3+.
- **Cache health:** cache-hit accuracy (0.92) remains at or above cache-miss
  accuracy (0.88) — the W10 cache is a cost win, not a correctness risk.

---

## M2 status

Production-ready RAG: FastAPI service (W5) wrapping the full W6–W11 pipeline,
`pytest eval/` green against baseline thresholds, all 8 KPIs reported (6 numeric,
2 honest N/A), architecture consolidated in ADR v2 (M2). Ready to defend retrieval
+ KB-lifecycle decisions at DR #2.

**Next snapshot:** M3 (W18) — after Phase 3 adds tools/agents (KPI #5 becomes
real) and injection defenses.

**The honesty moment — KPI #4 reconciliation.** The milestone is where definitions get **locked**. W11 reported retrieval hit rate via RAGAS *context precision* (0.82) as a stand-in; from M2 it's measured to its canonical definition — *correct chunk in top-K* (0.90) — with context precision (0.85) kept as a *supporting* metric. That's a change of **definition, not a regression**, and it's documented in the snapshot and the ADR. Being explicit about it is exactly what a top score on "ADR quality" looks like.

## 4 · ADR v2 (M2) — consolidate every Phase-2 decision

The milestone ADR puts every Phase-2 decision in **one table**: chunker, BM25 weight, reranker, top-K, cache threshold, tombstones, RAGAS thresholds, KPIs — each with its rationale. Mark it **v2 (M2)**. Here's the shape (yours will be fuller):

In [5]:
display(Markdown(Path('docs/adr/0001-capstone-framing.md').read_text()))

# ADR 0001 — Capstone Framing — v2 (M2)

*(Prior W1–W11 entries elided — keep your repo's full text. This is the Phase-2
consolidation the milestone asks for: every decision in one table.)*

## Phase 2 consolidated decisions (v2 · M2)

| Area | Decision | Why |
|---|---|---|
| Chunking | 512-token, 64 overlap | balances recall vs precision |
| Hybrid weight | BM25 0.4 + dense | catches exact-term queries |
| Reranker + top-K | cross-encoder, top-K 3 | lifts context precision; small prompt |
| Cache | semantic (0.95) + tombstones | cost win, safe deletes |
| Eval | RAGAS thresholds (faithfulness 0.85, recall 0.80, ...) | defensible bar |
| KPIs | 8-KPI dashboard, kb_version-tagged | portfolio evidence |

Status: **v2 (M2)** — Phase 2 closed; retrieval + KB-lifecycle decisions consolidated.


## 5 · The DR #2 one-pager — defend retrieval + KB lifecycle

DR #2's scope is your **retrieval + KB-lifecycle** decisions. The one-pager has five sections, each mapping to a rubric dimension. Here's the worked example:

In [6]:
display(Markdown(Path('docs/dr/dr2-summary-EXAMPLE.md').read_text()))

# DR #2 — EXAMPLE — Enterprise Knowledge Assistant — Retrieval + KB Lifecycle Defense

> Filled example for the capstone use case. Yours will differ; match the depth.

## 1. Problem framing
The assistant answers employees' questions about HR/IT/Finance policies from an
internal document corpus. Constraints: answers must be **grounded** (no invented
policy), cheap at scale, and safe when a topic isn't covered. RAG is the right
approach: the policy corpus changes often and must be cited — fine-tuning would
bake in stale facts and can't cite sources.

## 2. Retrieval choices  (trade-offs with KPI evidence)
- **Hybrid, BM25 weight 0.4.** Dense retrieval alone missed exact-term queries
  (policy codes, section numbers); adding BM25 raised retrieval hit rate. 0.4 was
  the point where hit rate peaked without precision dropping.
- **Cross-encoder rerank, top-K = 3.** Reranking the top candidates lifted context
  precision from ~0.7 to 0.85; top-K = 3 keeps the prompt small (cost) while
  covering multi-hop answers after the S1 recall fix.
- **Rejected:** dense-only (missed exact terms); top-K = 5 (added ~600 prompt
  tokens for no measurable quality gain).

## 3. KB lifecycle  (failure handling)
- **Tombstone policy:** deletes set `deleted_at` and are filtered from retrieval;
  we never hard-delete, so eval runs stay reproducible.
- **Update workflow:** full re-ingest, tombstone the old chunks, bump
  `kb_version` — see `docs/runbooks/rag_debugging.md` and the W10 update runbook.
- **Cache invalidation:** stale-cache is a real failure mode (drill S5); cache
  entries are invalidated on `kb_version` change. Cache-hit accuracy (0.92) stays
  above cache-miss accuracy (0.88), so the cache isn't serving stale answers.

## 4. The 8-KPI dashboard  (KPI reporting, with trend)
Task success **0.92**, grounded **0.92**, hallucination **0.08**, retrieval hit
**0.90**, cost **$0.000131**, latency **184/312 ms**; tool success and escalation
**N/A** (arrive W13 / W17). Full wk6→wk12 trajectory in `docs/kpi/wk12-snapshot.md`
— quality climbed across the build weeks and again at wk12 after a retrieval fix;
cost/latency dropped an order of magnitude at wk10.

## 5. Open trade-offs & questions
- **Open trade-off:** the semantic-cache threshold (0.95) was tuned on only 20
  golden questions; at real traffic I'd tune it against a measured false-hit rate.
- **Open question for reviewers:** is 20 golden questions enough to trust the KPI
  trend, or should the golden set grow before Phase 3 adds tool-use complexity?

### The 5-dimension Design Review rubric (each scored 1–4)

| Dimension | A '4' looks like |
|---|---|
| Problem framing | problem + context + constraints + why this approach wins |
| Trade-off articulation | trade-offs weighed with **KPI data** + alternatives rejected |
| KPI reporting | all 8 KPIs + trend vs last review |
| Failure handling | multiple failure modes + defenses |
| ADR quality | a living artifact with evolution visible |

**A strong defense answers, with evidence:** *Why hybrid + rerank?* (show the KPI lift and the alternatives you rejected) · *Why these thresholds?* (BM25 weight, top-K, cache threshold — tuned against what?) · *How does the KB update?* (tombstones + `kb_version` + cache invalidation — show the runbook) · *Where does it still fail?* (name a failure mode and your defense). Expect peers to probe one decision deeply: *"What did top-K = 5 cost you vs 3?"*, *"Show me a query where hybrid beats dense."*, *"What happens when a doc is deleted?"*

## Wrap — you can debug it *and* defend it

DR #2 is **formative** — the first hard bar is DR #3 (W18, average ≥ 2.5); use the feedback to get there. Ship all five M2 items, bring your numbers, and defend your retrieval + KB-lifecycle decisions. Then **Phase 3 — Agents**: LLMs that *use tools* to take actions. A different shape with new failure modes, but the same debugging discipline (and KPI #5, tool success, goes live in W13).

*These notebooks are for **seeing and explaining** the work; in the repo, the eval lives as a pytest suite that gates every commit and the API is your running service — the notebook to understand it, the suite and service to run it.*